# Gaussian Mixture Models (GMM)

## Co jsou Gaussian Mixture Models?

Gaussian Mixture Models (GMM) je pravděpodobnostní model předpokládající, že všechna datová body jsou generována ze směsi několika Gaussovských (normálních) rozdělení s neznámými parametry. Na rozdíl od K-means, který přiřazuje každý bod jednoznačně do jednoho shluku, GMM přiřazuje pravděpodobnosti příslušnosti bodu ke každému shluku.

### Princip algoritmu GMM:

1. **Inicializace** - Parametry Gaussovských rozdělení (střední hodnoty, kovarianční matice a váhy) jsou inicializovány, často pomocí K-means
2. **Expectation (E-krok)** - Výpočet pravděpodobnosti příslušnosti každého bodu ke každému shluku
3. **Maximization (M-krok)** - Aktualizace parametrů Gaussovských rozdělení na základě vypočtených pravděpodobností
4. **Iterace** - Opakování kroků E a M dokud nedojde ke konvergenci

### Matematické vyjádření:

GMM modeluje data jako vážený součet $K$ Gaussovských rozdělení:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

kde:
- $\pi_k$ jsou váhy komponent (musí splňovat $\sum_{k=1}^{K}\pi_k = 1$ a $\pi_k \geq 0$)
- $\boldsymbol{\mu}_k$ je vektor středních hodnot $k$-té komponenty
- $\boldsymbol{\Sigma}_k$ je kovarianční matice $k$-té komponenty
- $\mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ je Gaussovská hustota pravděpodobnosti

### Výhody GMM oproti K-means:

- Flexibilnější tvar shluků (eliptické místo sférických)
- Měkké přiřazení bodů ke shlukům (pravděpodobnosti místo jednoznačného přiřazení)
- Zachycení nejistoty v přiřazení
- Možnost modelovat shluky různých velikostí a orientací

In [ ]:
# Import potřebných knihoven
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import ListedColormap
from scipy import linalg
import matplotlib as mpl
from sklearn.decomposition import PCA

# Nastavení vizuálního stylu
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
np.random.seed(42)

## 1. Jednoduchý příklad GMM na syntetických datech

Začneme vytvořením a vizualizací syntetických dat, na kterých předvedeme základní použití GMM.

In [ ]:
# Vytvoření syntetických dat se 3 shluky s různými velikostmi a orientacemi
n_samples = 1000
random_state = 42

# Generování náhodných bodů ze tří různých normálních rozdělení
cluster_1 = np.random.multivariate_normal([0, 0], [[1, 0], [0, 1]], int(0.3 * n_samples))
cluster_2 = np.random.multivariate_normal([5, 5], [[1, -0.7], [-0.7, 1]], int(0.4 * n_samples))
cluster_3 = np.random.multivariate_normal([0, 5], [[0.5, 0], [0, 2]], int(0.3 * n_samples))

# Spojení dat do jedné matice
X = np.vstack([cluster_1, cluster_2, cluster_3])
# Vytvoření skutečných labels pro pozdější vyhodnocení
y_true = np.hstack([np.zeros(cluster_1.shape[0]), 
                    np.ones(cluster_2.shape[0]), 
                    np.ones(cluster_3.shape[0]) * 2]).astype(int)

# Vizualizace dat
plt.figure(figsize=(10, 8))
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.8)
plt.colorbar(label='Skutečný shluk')
plt.title('Syntetická data se 3 Gaussovskými shluky', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.grid(True)
plt.show()

In [ ]:
# Aplikace GMM
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=random_state)
gmm_labels = gmm.fit_predict(X)

# Vizualizace výsledků
plt.figure(figsize=(10, 8))
plt.scatter(X[:, 0], X[:, 1], c=gmm_labels, cmap='viridis', alpha=0.8)
plt.colorbar(label='GMM shluk')
plt.title('Výsledky GMM shlukování', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.grid(True)
plt.show()

# Výpočet metrik pro vyhodnocení kvality shlukování
silhouette = silhouette_score(X, gmm_labels)
ari = adjusted_rand_score(y_true, gmm_labels)

print(f"Silhouette skóre: {silhouette:.3f} (vyšší je lepší, max=1)")
print(f"Adjusted Rand Index: {ari:.3f} (vyšší je lepší, max=1)")
print("Parametry GMM modelu:")
print(f"  Váhy komponent: {gmm.weights_}")
print(f"  Střední hodnoty komponent:")
for i, mean in enumerate(gmm.means_):
    print(f"    Komponenta {i}: {mean}")

## 2. Vizualizace GMM komponent

Nyní si ukážeme, jak vizualizovat jednotlivé Gaussovské komponenty modelu na datech.

In [ ]:
# Funkce pro vykreslení elips reprezentujících Gaussovské komponenty
def plot_gmm(gmm, X, ax=None, colors=None):
    ax = ax or plt.gca()
    if colors is None:
        colors = plt.cm.viridis(np.linspace(0, 1, gmm.n_components))
        
    # Vytvoření mesh gridu pro vykreslení pravděpodobností
    x = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
    y = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 100)
    XX, YY = np.meshgrid(x, y)
    XY = np.column_stack([XX.ravel(), YY.ravel()])
    
    # Výpočet celkové log-pravděpodobnosti
    Z = -gmm.score_samples(XY)
    Z = Z.reshape(XX.shape)
    
    # Vykreslení kontur log-pravděpodobnosti
    ax.contour(XX, YY, Z, levels=10, linewidths=1, cmap='Greys_r')
    
    # Vykreslení bodů a jejich přiřazení ke shlukům
    labels = gmm.predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.8)
    
    # Vykreslení elips pro každou komponentu (95% konfidenční elipsy)
    for i, (weight, mean, covar) in enumerate(zip(gmm.weights_, gmm.means_, gmm.covariances_)):
        # Vlastní čísla a vlastní vektory
        v, w = linalg.eigh(covar)
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)  # 2 sigma (95%)
        u = w[0] / linalg.norm(w[0])
        
        # Úhel elipsy
        angle = np.arctan2(u[1], u[0])
        angle = 180.0 * angle / np.pi  # převod na stupně
        
        # Vykreslení elipsy
        ell = mpl.patches.Ellipse(mean, v[0], v[1], angle=180.0 + angle, 
                                  edgecolor=colors[i], facecolor='none', linewidth=2)
        ell.set_alpha(0.8)
        ax.add_patch(ell)
        ax.text(mean[0], mean[1], f"{i+1}", fontsize=12, 
                fontweight='bold', ha='center', va='center', color='red')
    
    # Přidání barevné škály
    return ax

# Vykreslení GMM komponent
plt.figure(figsize=(12, 10))
ax = plt.gca()
plot_gmm(gmm, X, ax)
plt.colorbar(plt.cm.ScalarMappable(cmap='viridis'), label='Cluster')
plt.title('GMM komponenty a přiřazení dat ke shlukům', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.grid(True)
plt.show()

## 3. Porovnání různých typů kovariančních matic v GMM

GMM umožňuje specifikovat různé typy kovariančních matic, což má vliv na tvar a orientaci shluků.

In [ ]:
# Různé typy kovariančních matic
covariance_types = ['spherical', 'diag', 'tied', 'full']

# Vykreslení výsledků pro různé typy kovariančních matic
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, covariance_type in enumerate(covariance_types):
    # Vytvoření a aplikace GMM s daným typem kovarianční matice
    gmm = GaussianMixture(n_components=3, covariance_type=covariance_type, random_state=random_state)
    gmm_labels = gmm.fit_predict(X)
    
    # Měření kvality shlukování
    silhouette = silhouette_score(X, gmm_labels)
    ari = adjusted_rand_score(y_true, gmm_labels)
    
    # Vykreslení výsledků
    plot_gmm(gmm, X, ax=axes[i])
    axes[i].set_title(f"Typ kovariance: {covariance_type}\nSilhouette: {silhouette:.3f}, ARI: {ari:.3f}", fontsize=12)
    axes[i].set_xlabel('První příznak', fontsize=10)
    axes[i].set_ylabel('Druhý příznak', fontsize=10)

plt.tight_layout()
plt.show()

# Vysvětlení typů kovariančních matic
print("Typy kovariančních matic v GMM:")
print("  'full': Každý shluk má svou vlastní obecnou kovarianční matici. Nejflexibilnější, ale nejvíce parametrů.")
print("  'tied': Všechny shluky sdílejí stejnou obecnou kovarianční matici.")
print("  'diag': Každý shluk má svou vlastní diagonální kovarianční matici (osy elipsy jsou rovnoběžné s osami).")
print("  'spherical': Každý shluk má svou vlastní jednoparametrickou kovarianční matici (kruhy místo elips).")

## 4. GMM na různých typech dat

Ukážeme, jak si GMM poradí s různými tvary dat ve srovnání s K-means.

In [ ]:
# Funkce pro vytvoření různých datových sad
def make_data():
    # 1. Normální Gaussovské shluky
    X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, 
                                  cluster_std=[1.0, 1.5, 0.5],
                                  random_state=42)
    
    # 2. Nelineární data - dva půlměsíce
    X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
    
    # 3. Shluky s různými velikostmi a orientacemi
    c1 = np.random.multivariate_normal([0, 0], [[0.2, 0], [0, 0.2]], 100)
    c2 = np.random.multivariate_normal([1, 1], [[1.5, -0.5], [-0.5, 0.5]], 200)
    X_varied = np.vstack([c1, c2])
    y_varied = np.hstack([np.zeros(100), np.ones(200)])
    
    return [(X_blobs, y_blobs, "Normální Gaussovské shluky"), 
            (X_moons, y_moons, "Nelineární půlměsíce"),
            (X_varied, y_varied, "Shluky s různými velikostmi")]

# Vytvoření dat
datasets = make_data()

# Srovnání GMM a K-means na různých typech dat
fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 15))

for i, (X, y, title) in enumerate(datasets):
    n_clusters = len(np.unique(y))
    
    # Původní data
    axes[i, 0].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.8)
    axes[i, 0].set_title(f"{title}\nPůvodní data", fontsize=12)
    axes[i, 0].set_xlabel('x', fontsize=10)
    axes[i, 0].set_ylabel('y', fontsize=10)
    
    # K-means
    km = KMeans(n_clusters=n_clusters, random_state=42)
    km_labels = km.fit_predict(X)
    km_score = adjusted_rand_score(y, km_labels)
    
    axes[i, 1].scatter(X[:, 0], X[:, 1], c=km_labels, cmap='viridis', alpha=0.8)
    axes[i, 1].set_title(f"K-means\nARI: {km_score:.3f}", fontsize=12)
    axes[i, 1].set_xlabel('x', fontsize=10)
    axes[i, 1].set_ylabel('y', fontsize=10)
    
    # GMM
    gmm = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=42)
    gmm_labels = gmm.fit_predict(X)
    gmm_score = adjusted_rand_score(y, gmm_labels)
    
    # Vykreslení GMM výsledků
    plot_gmm(gmm, X, ax=axes[i, 2])
    axes[i, 2].set_title(f"GMM (full)\nARI: {gmm_score:.3f}", fontsize=12)
    axes[i, 2].set_xlabel('x', fontsize=10)
    axes[i, 2].set_ylabel('y', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Určení optimálního počtu komponent v GMM

Pro určení optimálního počtu komponent v GMM můžeme použít několik metod:
1. Bayesovské informační kritérium (BIC)
2. Akaikeho informační kritérium (AIC)
3. Silhouette skóre

In [ ]:
# Použijeme data z prvního příkladu
X_sample = X

# Vypočteme BIC a AIC pro různý počet komponent
n_components_range = range(1, 10)
bic = []
aic = []
silhouette_scores = []

for n_components in n_components_range:
    # Vytvoření GMM
    gmm = GaussianMixture(n_components=n_components,
                         covariance_type='full',
                         random_state=42)
    # Trénování GMM
    gmm.fit(X_sample)
    
    # Výpočet BIC a AIC
    bic.append(gmm.bic(X_sample))
    aic.append(gmm.aic(X_sample))
    
    # Výpočet silhouette skóre (kromě případu s 1 komponentou)
    if n_components > 1:
        labels = gmm.predict(X_sample)
        silhouette_scores.append(silhouette_score(X_sample, labels))
    else:
        silhouette_scores.append(0)  # Pro jednu komponentu nemá silhouette smysl

# Vizualizace výsledků
plt.figure(figsize=(15, 5))

plt.subplot(131)
plt.plot(n_components_range, bic, 'o-', label='BIC')
plt.xlabel('Počet komponent')
plt.ylabel('BIC skóre')
plt.title('Bayesovské informační kritérium (BIC)')
plt.grid(True)
plt.xticks(n_components_range)

plt.subplot(132)
plt.plot(n_components_range, aic, 'o-', label='AIC')
plt.xlabel('Počet komponent')
plt.ylabel('AIC skóre')
plt.title('Akaikeho informační kritérium (AIC)')
plt.grid(True)
plt.xticks(n_components_range)

plt.subplot(133)
plt.plot(n_components_range, silhouette_scores, 'o-', label='Silhouette')
plt.xlabel('Počet komponent')
plt.ylabel('Silhouette skóre')
plt.title('Silhouette skóre')
plt.grid(True)
plt.xticks(n_components_range)

plt.tight_layout()
plt.show()

# Identifikace optimálního počtu komponent
best_bic_idx = np.argmin(bic)
best_aic_idx = np.argmin(aic)
best_silhouette_idx = np.argmax(silhouette_scores)

print(f"Optimální počet komponent podle BIC: {n_components_range[best_bic_idx]}")
print(f"Optimální počet komponent podle AIC: {n_components_range[best_aic_idx]}")
print(f"Optimální počet komponent podle Silhouette: {n_components_range[best_silhouette_idx]}")

## 6. GMM pro detekci anomálií

GMM lze efektivně využít pro detekci anomálií, kde body s nízkou pravděpodobností podle modelu jsou považovány za anomálie.

In [ ]:
# Vytvoření dat s anomáliemi
np.random.seed(42)
# Normální data
X_normal = np.random.multivariate_normal([2, 2], [[1, 0.5], [0.5, 1]], 300)
# Anomálie
X_anomalies = np.random.uniform(low=[-2, -2], high=[6, 6], size=(30, 2))
# Kombinace dat
X_mixed = np.vstack([X_normal, X_anomalies])

# Aplikace GMM pro detekci anomálií
gmm_anomaly = GaussianMixture(n_components=1, covariance_type='full', random_state=42)
gmm_anomaly.fit(X_mixed)

# Výpočet log-pravděpodobností jednotlivých bodů
log_probs = gmm_anomaly.score_samples(X_mixed)

# Určení prahu pro anomálie (například dolních 5% bodů podle pravděpodobnosti)
threshold = np.percentile(log_probs, 5)

# Identifikace anomálií
is_anomaly = log_probs < threshold

# Vizualizace výsledků
plt.figure(figsize=(12, 10))

# Vytvoření mesh gridu pro vykreslení hustoty pravděpodobnosti
x = np.linspace(X_mixed[:, 0].min() - 1, X_mixed[:, 0].max() + 1, 100)
y = np.linspace(X_mixed[:, 1].min() - 1, X_mixed[:, 1].max() + 1, 100)
XX, YY = np.meshgrid(x, y)
XY = np.column_stack([XX.ravel(), YY.ravel()])

# Výpočet hustoty pravděpodobnosti na mesh gridu
Z = gmm_anomaly.score_samples(XY)
Z = Z.reshape(XX.shape)

# Vykreslení kontur hustoty pravděpodobnosti
plt.contourf(XX, YY, Z, levels=50, cmap='viridis', alpha=0.5)
plt.colorbar(label='Log-pravděpodobnost')

# Vykreslení bodů
plt.scatter(X_mixed[:, 0], X_mixed[:, 1], c='white', edgecolor='black', alpha=0.6)
# Zvýraznění anomálií
plt.scatter(X_mixed[is_anomaly, 0], X_mixed[is_anomaly, 1], c='red', s=80, label='Anomálie')

plt.title('Detekce anomálií pomocí GMM', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

# Statistiky
print(f"Detekováno {is_anomaly.sum()} anomálií z celkových {len(X_mixed)} bodů")
print(f"Procento anomálií: {is_anomaly.sum() / len(X_mixed) * 100:.1f}%")

## 7. GMM na reálných datech

Aplikujme GMM na Iris dataset, který je standardně používán pro testování shlukovacích algoritmů.

In [ ]:
# Načtení Iris datasetu
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Standardizace příznaků
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Aplikace GMM
gmm_iris = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
y_pred_iris = gmm_iris.fit_predict(X_iris_scaled)

# Výpočet metrik výkonnosti
silhouette_iris = silhouette_score(X_iris_scaled, y_pred_iris)
ari_iris = adjusted_rand_score(y_iris, y_pred_iris)

# Vizualizace výsledků pomocí PCA
pca = PCA(n_components=2)
X_iris_pca = pca.fit_transform(X_iris_scaled)

plt.figure(figsize=(15, 6))

plt.subplot(121)
plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris, cmap='viridis', edgecolor='k')
plt.title('Skutečné třídy Iris datasetu', fontsize=14)
plt.xlabel('První hlavní komponenta', fontsize=12)
plt.ylabel('Druhá hlavní komponenta', fontsize=12)
plt.colorbar(label='Třída')
plt.grid(True)

plt.subplot(122)
plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_pred_iris, cmap='viridis', edgecolor='k')
plt.title(f'GMM clustering (ARI: {ari_iris:.3f}, Silhouette: {silhouette_iris:.3f})', fontsize=14)
plt.xlabel('První hlavní komponenta', fontsize=12)
plt.ylabel('Druhá hlavní komponenta', fontsize=12)
plt.colorbar(label='Cluster')
plt.grid(True)

plt.tight_layout()
plt.show()

# Zobrazení detailů o shlucích a jejich interpretace
print(f"Adjusted Rand Index: {ari_iris:.3f} (vyšší hodnoty značí lepší shodu se skutečnými třídami)")
print(f"Silhouette skóre: {silhouette_iris:.3f}")
print("\nVáhy jednotlivých komponent GMM:")
for i, weight in enumerate(gmm_iris.weights_):
    print(f"  Komponenta {i+1}: {weight:.3f}")

# Vizualizace skutečných tříd vs. predikovaných shluků
from sklearn.metrics import confusion_matrix
import pandas as pd

# Normalizace confusion matrix na procentuální hodnoty
conf_mat = confusion_matrix(y_iris, y_pred_iris)
conf_mat_norm = conf_mat.astype('float') / conf_mat.sum(axis=1)[:, np.newaxis]

# Vytvoření DataFrame pro lepší vizualizaci
df_conf = pd.DataFrame(conf_mat_norm, 
                       index=[f'Třída {i}' for i in range(3)],
                       columns=[f'Shluk {i}' for i in range(3)])

plt.figure(figsize=(10, 8))
sns.heatmap(df_conf, annot=True, cmap='Blues', fmt='.2f')
plt.title('Normalizovaná Confusion Matrix: Skutečné třídy vs. GMM shluky', fontsize=14)
plt.xlabel('Predikované shluky (GMM)', fontsize=12)
plt.ylabel('Skutečné třídy', fontsize=12)
plt.show()

## 8. Varianty GMM: Bayesovské GMM

Scikit-learn poskytuje také Bayesovské varianty GMM, které mohou automaticky určit počet komponent.

In [ ]:
from sklearn.mixture import BayesianGaussianMixture

# Použijeme data s 3 shluky
X_sample = X  # Data z prvního příkladu

# Aplikace Bayesovského GMM s různými hodnotami koncentračního prioru
weight_concentrations = [0.01, 0.1, 1, 10]
fig, axes = plt.subplots(1, len(weight_concentrations), figsize=(20, 5))

for i, weight_concentration in enumerate(weight_concentrations):
    # Inicializace modelu s více komponentami než je skutečně potřeba
    bgmm = BayesianGaussianMixture(
        n_components=10, 
        weight_concentration_prior=weight_concentration,
        max_iter=500,
        random_state=42
    )
    
    # Trénování modelu
    bgmm.fit(X_sample)
    
    # Predikce shluků
    labels = bgmm.predict(X_sample)
    
    # Vykreslení výsledků
    axes[i].scatter(X_sample[:, 0], X_sample[:, 1], c=labels, cmap='viridis', alpha=0.8)
    
    # Vykreslení pouze komponent s významnou vahou
    active_components = np.where(bgmm.weights_ > 0.01)[0]
    
    for j, idx in enumerate(active_components):
        covar = bgmm.covariances_[idx]
        mean = bgmm.means_[idx]
        
        # Vlastní čísla a vektory
        v, w = linalg.eigh(covar)
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)  # 2 sigma
        u = w[0] / linalg.norm(w[0])
        
        # Úhel elipsy
        angle = np.arctan2(u[1], u[0])
        angle = 180.0 * angle / np.pi  # převod na stupně
        
        # Vykreslení elipsy
        color = plt.cm.viridis(float(j) / len(active_components))
        ell = mpl.patches.Ellipse(mean, v[0], v[1], angle=180.0 + angle, 
                                  edgecolor=color, facecolor='none', linewidth=2)
        axes[i].add_patch(ell)
    
    axes[i].set_title(f"Weight concentration prior: {weight_concentration}"
                     f"\nAktivní komponenty: {len(active_components)}", fontsize=12)
    axes[i].set_xlabel('První příznak', fontsize=10)
    axes[i].set_ylabel('Druhý příznak', fontsize=10)
    axes[i].grid(True)

plt.tight_layout()
plt.show()

## 9. Shrnutí GMM

### Výhody Gaussian Mixture Models:
1. **Flexibilita v modelování shluků** - GMM umožňuje modelovat shluky různých velikostí, tvarů a orientací, což je výhoda oproti například K-means.
2. **Soft clustering** - GMM poskytuje pravděpodobnosti příslušnosti ke shlukům, což umožňuje lépe zachytit nejistotu.
3. **Pravděpodobnostní model** - GMM je generativní model, což umožňuje generovat nové vzorky a odhadnout hustotu pravděpodobnosti.
4. **Detekce anomálií** - GMM lze použít pro detekci anomálií pomocí pravděpodobností.
5. **Teoretický základ** - GMM má solidní matematický základ a vazby na EM (Expectation-Maximization) algoritmus.

### Nevýhody GMM:
1. **Citlivost na inicializaci** - Výsledky GMM mohou záviset na počáteční inicializaci parametrů.
2. **Nutnost specifikace počtu komponent** - Podobně jako K-means, GMM vyžaduje předem znát počet shluků.
3. **Předpoklad Gaussovského rozdělení** - GMM předpokládá, že data pochází ze směsi Gaussových rozdělení, což nemusí vždy platit.
4. **Výpočetní náročnost** - GMM může být výpočetně náročnější než jednodušší algoritmy jako K-means, zvláště pro vysokodimenzionální data.
5. **Riziko přetrénování** - GMM s mnoha komponentami může vést k přetrénování, zejména při malém množství dat.

### Klíčové parametry při použití GMM:
1. **n_components** - Počet Gaussovských komponent (shluků).
2. **covariance_type** - Typ kovarianční matice ('full', 'tied', 'diag', 'spherical').
3. **max_iter** - Maximální počet iterací EM algoritmu.
4. **tol** - Tolerance pro konvergenci EM algoritmu.
5. **random_state** - Seed pro reprodukovatelnost výsledků.
6. **init_params** - Metoda inicializace ('kmeans' nebo 'random').

### Typické aplikace GMM:
- Segmentace obrazu a videa
- Modelování přirozeného jazyka a rozpoznávání řeči
- Segmentace zákazníků v marketingu
- Detekce anomálií a odlehlých hodnot
- Doplňování chybějících hodnot
- Kvantizace vektorů v kompresi dat
- Redukce dimenzionality a vizualizace dat